# 🚀 Обучение LoRA персонажа ALISA_HERO

**Инструкция — 3 действия:**
1. Перетащите файл **`dataset.zip`** в левую панель (📁 Файлы)
2. Нажмите **Среда выполнения ➔ Выполнить всё**
3. Через 5–6 минут ячейка №5 выведет ссылку на скачивание файла весов!

In [ ]:
# ЯЧЕЙКА 1: Проверка GPU и установка совместимых версий
import subprocess
import sys

print('=== GPU INFO ===')
subprocess.run(['nvidia-smi'], check=False)

# Удаляем несовместимый torchao и ставим проверенные версии
packages = [
    'torchao',
]
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y'] + packages, check=False)

install = [
    'diffusers==0.30.3',
    'transformers==4.44.2',
    'peft==0.13.2',
    'accelerate==0.34.2',
    'safetensors',
    'bitsandbytes',
    'torchvision',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + install, check=True)
print('OK: All packages installed successfully!')

In [ ]:
# ЯЧЕЙКА 2: Проверка версий на совместимость
import importlib
ok = True
for pkg, expected in [('diffusers', '0.30.3'), ('peft', '0.13.2'), ('transformers', '4.44.2')]:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', 'unknown')
        status = 'OK' if expected in ver else 'WARN (unexpected version)'
        print(f'  {pkg}: {ver}  [{status}]')
    except Exception as e:
        print(f'  {pkg}: IMPORT ERROR - {e}')
        ok = False

# Verify critical import that failed before
try:
    from diffusers.loaders.peft import PeftAdapterMixin
    print('  diffusers.loaders.peft: OK')
except Exception as e:
    print(f'  diffusers.loaders.peft: ERROR - {e}')
    ok = False

if ok:
    print('\nAll version checks passed! Proceeding to training...')
else:
    raise RuntimeError('Version conflict detected. Restart the runtime and run all cells again.')

In [ ]:
# ЯЧЕЙКА 3: Распаковка датасета
import os
import zipfile
import shutil

if not os.path.exists('/content/dataset.zip'):
    raise FileNotFoundError(
        'Файл dataset.zip не найден!\n'
        'Перетащите его в левую панель (📁 Файлы) и запустите эту ячейку снова.'
    )

shutil.rmtree('/content/dataset_images', ignore_errors=True)
os.makedirs('/content/dataset_images', exist_ok=True)

with zipfile.ZipFile('/content/dataset.zip', 'r') as z:
    z.extractall('/content/dataset_images')

# Find all images recursively (handles nested folders in zip)
all_images = []
for root, dirs, files in os.walk('/content/dataset_images'):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append(os.path.join(root, f))

print(f'Found {len(all_images)} training images:')
for img in all_images:
    size_kb = os.path.getsize(img) // 1024
    print(f'  {os.path.basename(img)} ({size_kb} KB)')

if len(all_images) == 0:
    raise ValueError('No images found in dataset.zip! Check the archive contents.')

In [ ]:
# ЯЧЕЙКА 4: Загрузка скрипта обучения v0.30.3 и запуск LoRA DreamBooth
import os
import shutil
import subprocess

shutil.rmtree('/content/alisa_lora_output', ignore_errors=True)
os.makedirs('/content/alisa_lora_output', exist_ok=True)

# Download the exact matching training script for diffusers v0.30.3
script_url = 'https://raw.githubusercontent.com/huggingface/diffusers/v0.30.3/examples/dreambooth/train_dreambooth_lora.py'
script_path = '/content/train_dreambooth_lora.py'
result = subprocess.run(['wget', '-q', '-O', script_path, script_url], capture_output=True)
if result.returncode != 0 or not os.path.exists(script_path):
    raise RuntimeError(f'Failed to download training script: {result.stderr.decode()}')

print(f'Training script downloaded ({os.path.getsize(script_path)//1024} KB)')
print('Starting LoRA training (500 steps, ~5 minutes)...\n')

cmd = [
    'python', script_path,
    '--pretrained_model_name_or_path', 'runwayml/stable-diffusion-v1-5',
    '--instance_data_dir', '/content/dataset_images',
    '--output_dir', '/content/alisa_lora_output',
    '--instance_prompt', 'a photo of ALISA_HERO woman, vibrant red hair, warm amber eyes',
    '--resolution', '512',
    '--train_batch_size', '1',
    '--gradient_accumulation_steps', '1',
    '--learning_rate', '1e-4',
    '--lr_scheduler', 'constant',
    '--lr_warmup_steps', '0',
    '--max_train_steps', '500',
    '--mixed_precision', 'fp16',
    '--seed', '42',
]

result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError('Training failed! Check error output above.')

print('\nTraining completed! Checking output...')
out_files = []
for root, dirs, fnames in os.walk('/content/alisa_lora_output'):
    for f in fnames:
        out_files.append(os.path.join(root, f))
print(f'Output files ({len(out_files)} total):', [os.path.basename(f) for f in out_files])

In [ ]:
# ЯЧЕЙКА 5: Упаковка и публикация ссылки для скачивания
import os
import shutil
import subprocess

lora_file = None
for root, dirs, fnames in os.walk('/content/alisa_lora_output'):
    for f in fnames:
        if f.endswith('.safetensors') or f.endswith('.bin'):
            lora_file = os.path.join(root, f)
            break

if lora_file is None:
    print('ERROR: No .safetensors file found!')
    print('All output files:', os.listdir('/content/alisa_lora_output'))
    raise FileNotFoundError('LoRA weights not found. Training may have failed.')

size_mb = os.path.getsize(lora_file) / (1024 * 1024)
print(f'LoRA weights found: {lora_file} ({size_mb:.1f} MB)')

# Copy to root with clean name
shutil.copy(lora_file, '/content/alisa_lora.safetensors')

# Create bundle zip
bundle_path = '/content/alisa_lora_bundle.zip'
subprocess.run(['zip', '-r', bundle_path, 'alisa_lora.safetensors'], cwd='/content', check=True)
bundle_mb = os.path.getsize(bundle_path) / (1024 * 1024)
print(f'Bundle created: {bundle_mb:.1f} MB')

# Upload and get download link
print('\nUploading to tmpfiles.org...')
result = subprocess.run(
    ['curl', '-F', f'file=@{bundle_path}', 'https://tmpfiles.org/api/v1/upload'],
    capture_output=True, text=True
)
response = result.stdout.strip()
print('\n' + '='*60)
print('TRAINING COMPLETE!')
print('Download your LoRA model here:')
print(response)
print('='*60)
print('\nPaste the link above to your assistant to connect the model!')